# YOLOv11s-seg Segmentation — IRRG

Trains, validates, and tests a YOLOv11s-seg instance-segmentation model on IRRG 512×512 patches (0 % overlap). Colour augmentation is disabled. Cosine LR scheduling is used over 200 epochs. Retina masks and non-overlapping instance masks improve crown boundary accuracy.

**Install dependencies**

In [ ]:
# Install only once
!pip install ultralytics

**Import libraries**

In [ ]:
import json
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
from IPython.display import display, Image
from shapely.geometry import Polygon, mapping
from ultralytics import YOLO

random.seed(42)

# Locate project root (walks upward until the 'src' package is found)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import PROJECT_ROOT

**Define paths and run name**

In [ ]:
# Path to the YOLO data configuration file that lists the train/val/test
# image directories and class names
YAML_DIR = PROJECT_ROOT / "configs/yolov11s_instance_segmentation_rgb_config_v01.yaml"

# Root directory where Ultralytics creates the run sub-folder.
# After training the structure will be:
#   <SAVE_DIR> / <RUN_NAME> / weights / best.pt
SAVE_DIR = PROJECT_ROOT / "models" / "yolov11_segmentation"

# Name for this specific run
RUN_NAME = "irrg"

**Load pre-trained YOLO segmentation weights**

In [ ]:
model = YOLO("yolo11s-seg.pt")

**Training function**

In [ ]:
def yolo_train(
    model,
    data_yaml: str,
    epochs: int,
    imgsz: int,
    batch: int,
    name: str,
    project: str,
    overlap_mask: bool,
    retina_masks: bool,
    single_cls: bool,
    cos_lr: bool,
    auto_augment,
    erasing: float,
    mosaic: float,
    device: int,
    workers: int,
    box: float,
    cls: float,
    max_retries: int = 5,
) -> None:
    """
    Train a YOLO segmentation model on IRRG imagery with an automatic retry
    mechanism and explicit control over mask quality and augmentation settings.

    retina_masks=True produces higher-resolution masks by up-sampling predictions
    to the original image size before loss computation, which improves boundary
    accuracy for tree crowns.  overlap_mask=False prevents overlapping instance
    masks from being merged, which is important when crowns are adjacent.
    cos_lr=True uses a cosine learning-rate schedule with 200 epochs, which
    tends to improve final accuracy over a step schedule.

    Args:
        model:        Loaded YOLO segmentation model instance.
        data_yaml:    Path to the Ultralytics data config (.yaml).
        epochs:       Total training epochs.
        imgsz:        Input image size (square, in pixels).
        batch:        Batch size.
        name:         Sub-folder name for this run inside 'project'.
        project:      Root output directory (SAVE_DIR).
        overlap_mask: Allow overlapping instance masks (False = no overlap).
        retina_masks: Use full-resolution mask head (True = better boundaries).
        single_cls:   Treat all classes as one foreground class.
        cos_lr:       Use cosine learning-rate schedule.
        auto_augment: AutoAugment policy name, or None to disable.
        erasing:      Random-erasing probability (0.0 = disabled).
        mosaic:       Mosaic augmentation probability (0.0 = disabled).
        device:       GPU index or 'cpu'.
        workers:      DataLoader worker processes.
        box:          Box-loss weight.
        cls:          Class-loss weight.
        max_retries:  Maximum recovery attempts on OSError.
    """
    def _log_gpu(prefix: str = "") -> None:
        if torch.cuda.is_available():
            mb = torch.cuda.memory_allocated(device) / (1024 ** 2)
            print(f"{prefix} GPU memory allocated: {mb:.2f} MB")

    _log_gpu("[Before training]")

    for attempt in range(max_retries):
        try:
            start = time.time()
            model.train(
                data=data_yaml,
                epochs=epochs,
                imgsz=imgsz,
                batch=batch,
                name=name,
                project=project,
                overlap_mask=overlap_mask,
                retina_masks=retina_masks,
                single_cls=single_cls,
                cos_lr=cos_lr,
                auto_augment=auto_augment,
                erasing=erasing,
                mosaic=mosaic,
                device=device,
                workers=workers,
                box=box,
                cls=cls,
            )
            elapsed = time.time() - start
            print(f"\nTraining complete in {elapsed:.0f} s")
            _log_gpu("[After training]")
            return

        except OSError as exc:
            print(f"OSError on attempt {attempt + 1}/{max_retries}: {exc}")
            if attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)
                print(f"Retrying in {wait} s …")
                time.sleep(wait)
            else:
                print("Max retries reached — training aborted.")

        except Exception as exc:
            print(f"Unexpected error: {exc}")
            break

**Run training**

In [ ]:
yolo_train(
    model,
    data_yaml    = str(YAML_DIR),
    epochs       = 200,
    imgsz        = 640,
    batch        = 16,
    name         = RUN_NAME,
    project      = str(SAVE_DIR),
    overlap_mask = False,
    retina_masks = True,
    single_cls   = True,
    cos_lr       = True,
    auto_augment = None,
    erasing      = 0.0,
    mosaic       = 0.0,
    device       = 0,
    workers      = 0,
    box          = 12.0,
    cls          = 2.0,
)

**Load best checkpoint**

In [ ]:
# The best checkpoint is always saved at <SAVE_DIR>/<RUN_NAME>/weights/best.pt.
# Using RUN_DIR means you only need to change SAVE_DIR / RUN_NAME at the top
# of the notebook — no path duplication here.
model = YOLO(SAVE_DIR / RUN_NAME / "weights" / "best.pt")

**Validate on val split**

In [ ]:
results = model.val(
    data=str(YAML_DIR),
    split="val",
    plots=True,
    save_json=True,
)

# Ultralytics segmentation results expose separate .box and .seg metric objects.
# We extract per-class values via getattr to handle models that may not populate
# every attribute (e.g. single-class runs).
names            = results.names
box_p            = getattr(results.box, "p",      None)
box_r            = getattr(results.box, "r",      None)
box_map50_percls = getattr(results.box, "map50s", None)
box_map_percls   = getattr(results.box, "maps",   None)
seg_p            = getattr(results.seg, "p",      None)
seg_r            = getattr(results.seg, "r",      None)
seg_map50_percls = getattr(results.seg, "map50s", None)
seg_map_percls   = getattr(results.seg, "maps",   None)

rows = []
for i, name in names.items():
    rows.append({
        "Class":          name,
        "Box_Precision":  box_p[i]            if box_p            is not None else None,
        "Box_Recall":     box_r[i]            if box_r            is not None else None,
        "Box_mAP50":      box_map50_percls[i] if box_map50_percls is not None else results.box.map50,
        "Box_mAP50-95":   box_map_percls[i]   if box_map_percls   is not None else results.box.map,
        "Mask_Precision": seg_p[i]            if seg_p            is not None else None,
        "Mask_Recall":    seg_r[i]            if seg_r            is not None else None,
        "Mask_mAP50":     seg_map50_percls[i] if seg_map50_percls is not None else results.seg.map50,
        "Mask_mAP50-95":  seg_map_percls[i]   if seg_map_percls   is not None else results.seg.map,
    })

df_val = pd.DataFrame(rows)
print("Validation metrics:\n")
print(df_val)

out_csv = SAVE_DIR / RUN_NAME / "val_metrics.csv"
df_val.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

**Plot training loss curves**

In [ ]:
results_csv = SAVE_DIR / RUN_NAME / "results.csv"
df = pd.read_csv(results_csv, on_bad_lines="skip")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["train/box_loss"], label="Train Box Loss")
ax.plot(df["epoch"], df["val/box_loss"],   label="Val Box Loss",  linestyle="--")
ax.plot(df["epoch"], df["train/seg_loss"], label="Train Seg Loss")
ax.plot(df["epoch"], df["val/seg_loss"],   label="Val Seg Loss",  linestyle="--")
ax.plot(df["epoch"], df["train/cls_loss"], label="Train Cls Loss")
ax.plot(df["epoch"], df["val/cls_loss"],   label="Val Cls Loss",  linestyle="--")
ax.plot(df["epoch"], df["train/dfl_loss"], label="Train DFL Loss")
ax.plot(df["epoch"], df["val/dfl_loss"],   label="Val DFL Loss",  linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training vs Validation Loss")
ax.legend()
ax.grid(True)
fig.tight_layout()
fig.savefig(SAVE_DIR / RUN_NAME / "loss_curves.png", dpi=150)
plt.show()

**Evaluate on test split**

In [ ]:
results = model.val(
    data=str(YAML_DIR),
    split="test",
    plots=True,
    save_json=True,
)

names            = results.names
box_p            = getattr(results.box, "p",      None)
box_r            = getattr(results.box, "r",      None)
box_map50_percls = getattr(results.box, "map50s", None)
box_map_percls   = getattr(results.box, "maps",   None)
seg_p            = getattr(results.seg, "p",      None)
seg_r            = getattr(results.seg, "r",      None)
seg_map50_percls = getattr(results.seg, "map50s", None)
seg_map_percls   = getattr(results.seg, "maps",   None)

rows = []
for i, name in names.items():
    rows.append({
        "Class":          name,
        "Box_Precision":  box_p[i]            if box_p            is not None else None,
        "Box_Recall":     box_r[i]            if box_r            is not None else None,
        "Box_mAP50":      box_map50_percls[i] if box_map50_percls is not None else results.box.map50,
        "Box_mAP50-95":   box_map_percls[i]   if box_map_percls   is not None else results.box.map,
        "Mask_Precision": seg_p[i]            if seg_p            is not None else None,
        "Mask_Recall":    seg_r[i]            if seg_r            is not None else None,
        "Mask_mAP50":     seg_map50_percls[i] if seg_map50_percls is not None else results.seg.map50,
        "Mask_mAP50-95":  seg_map_percls[i]   if seg_map_percls   is not None else results.seg.map,
    })

df_test = pd.DataFrame(rows)
print("Test metrics:\n")
print(df_test)

out_csv = SAVE_DIR / RUN_NAME / "test_metrics.csv"
df_test.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

**Per-class performance bar chart (test set)**

In [ ]:
df = pd.read_csv(SAVE_DIR / RUN_NAME / "test_metrics.csv")
metrics = [
    "Box_Precision", "Box_Recall", "Box_mAP50", "Box_mAP50-95",
    "Mask_Precision", "Mask_Recall", "Mask_mAP50", "Mask_mAP50-95",
]
n_classes = len(df)
bar_width  = 0.10
x          = np.arange(n_classes)

fig, ax = plt.subplots(figsize=(14, 7))
for i, metric in enumerate(metrics):
    ax.bar(x + i * bar_width, df[metric], width=bar_width, label=metric)

ax.set_xticks(x + bar_width * (len(metrics) - 1) / 2)
ax.set_xticklabels(df["Class"], rotation=45, ha="right")
ax.set_ylabel("Metric Value")
ax.set_title("Per-Class Performance — Test Set")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.6)
fig.tight_layout()
fig.savefig(SAVE_DIR / RUN_NAME / "test_metrics_per_class.png", dpi=300)
plt.show()